## Creating and Using both In-Built as well as Custom Tools

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.tools.retriever import create_retriever_tool

from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper

import wikipedia

from dotenv import load_dotenv

C:\Users\viren\AppData\Local\Temp\ipykernel_25736\867601294.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Users\viren\generative-ai-implementations\langchain\agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
load_dotenv()

True

In [3]:
# Load LLM and Embeddings Model
model = ChatGroq(model="llama-3.1-8b-instant")
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11932.33it/s]


In [ ]:
wikipedia.set_user_agent("MyLangChainAgent/1.0 (pranayprasad7001l@gmail.com)")

In [8]:
# Used the in-built tool of wikipedia & Arxiv
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=300)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)

In [9]:
# Custom Tools [RAG Tools]
loader = WebBaseLoader(web_path="https://docs.langchain.com/oss/python/langchain/middleware/built-in")
docs = loader.load()
splitted_documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectorstore = FAISS.from_documents(splitted_documents, embedding)
retriever = vectorstore.as_retriever()

In [10]:
retriever_tool = create_retriever_tool(retriever, "langchain-middleware-search", "Search any info about langchain built-in middlewares")
retriever_tool.name

'langchain-middleware-search'

In [11]:
tools = [wiki, arxiv, retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\viren\\generative-ai-implementations\\langchain\\agent\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=300)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=300)),
 StructuredTool(name='langchain-middleware-search', description='Search any info about langchain built-in middlewares', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001CE0016D580>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001CE0016D300>)]

In [12]:
from langsmith import Client
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

client = Client()

prompt = client.pull_prompt(
    "hwchase17/openai-tools-agent", 
    dangerously_pull_public_prompt=True
)

agent = create_tool_calling_agent(model, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [13]:
agent_executor.invoke({"input": "tell me about langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langchain-middleware-search` with `{'query': 'langsmith'}`


Prebuilt middleware - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentJoin our product experts for a "Build More with LangSmith: Summer AMA Series" from July 9-August 12, 2026. RSVP todayDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationMiddlewarePrebuilt middlewareOverviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLon

{'input': 'tell me about langsmith',
 'output': 'The information about LangSmith was found in the documentation of LangChain.'}

In [14]:
print(wiki.run("Langchain"))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis a
